# Whisper Transcription (HTTP-only) in OpenShift

This notebook wraps a **minimal HTTP flow** for sending local audio files to your **OpenAI-compatible Whisper** service in OpenShift.

**What you get**
- Clear configuration section
- Small helper to list audio files
- A `requests`-based function that posts to `/v1/audio/transcriptions`
- Simple ipywidgets UI to pick, play, and transcribe files

> **Scope**: This version intentionally uses only the raw HTTP approach (no OpenAI SDK).

## Prerequisites
- Run this where the service DNS (e.g., `*.svc.cluster.local`) is reachable.
- Your Whisper service should implement `POST /v1/audio/transcriptions` (OpenAI-compatible).
- Put a few test audio files in the configured folder (default: `/opt/app-root/src/audio_data/`).
- If your endpoint requires auth, set an environment variable `OPENAI_API_KEY` before running.

In [1]:
# Optional: install dependencies if needed
# %pip install --quiet --upgrade requests ipywidgets
# If running in JupyterLab, enable widgets once (restart kernel might be required):
# %pip install --quiet jupyterlab-widgets ipywidgets
# %jupyter nbextension enable --py widgetsnbextension

## 1) Configuration
Update these values to match your environment and directory layout.

In [2]:
# === CONFIG ===
from pathlib import Path
import os

# Namespace (replace with your namespace!)
NAMESPACE = "whisper-quickstart"

# Whisper endpoint (adjust port/scheme if needed)
# WHISPER_HOST = f"http://whisper-large-v3-predictor.{NAMESPACE}.svc.cluster.local:8080"
# https://whisper-large-v3-predictor.whisper-quickstart.svc.cluster.local:8443
WHISPER_HOST = f"https://whisper-large-v3-whisper-quickstart.apps.cluster-d44sv.d44sv.sandbox2961.opentlc.com"

WHISPER_API = f"{WHISPER_HOST}/v1/audio/transcriptions"

# Model + auth (if any)
WHISPER_MODEL = "whisper-large-v3"
# Optional auth: set via env var OPENAI_API_KEY (or assign a token string here)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "eyJhbGciOiJSUzI1NiIsImtpZCI6IlNLM25MWXZfQVNWenV2RFdmUEsyTHZyWVplengzQTktWVRXZUVMZElrNkkifQ.eyJpc3MiOiJrdWJlcm5ldGVzL3NlcnZpY2VhY2NvdW50Iiwia3ViZXJuZXRlcy5pby9zZXJ2aWNlYWNjb3VudC9uYW1lc3BhY2UiOiJ3aGlzcGVyLXF1aWNrc3RhcnQiLCJrdWJlcm5ldGVzLmlvL3NlcnZpY2VhY2NvdW50L3NlY3JldC5uYW1lIjoiZGVmYXVsdC1uYW1lLXdoaXNwZXItd2hpc3Blci1sYXJnZS12My1zYSIsImt1YmVybmV0ZXMuaW8vc2VydmljZWFjY291bnQvc2VydmljZS1hY2NvdW50Lm5hbWUiOiJ3aGlzcGVyLWxhcmdlLXYzLXNhIiwia3ViZXJuZXRlcy5pby9zZXJ2aWNlYWNjb3VudC9zZXJ2aWNlLWFjY291bnQudWlkIjoiNGVlOTNlNDMtZTFhZi00NDliLWFkN2ItYzI4YjkwMzQ5NzM3Iiwic3ViIjoic3lzdGVtOnNlcnZpY2VhY2NvdW50OndoaXNwZXItcXVpY2tzdGFydDp3aGlzcGVyLWxhcmdlLXYzLXNhIn0.LltgP558GhgdT468UYMSOFkor2gR3fBrxoLilJDRKYEjA6FLYxp9SoTfemETCErH6iMWkt85cqVTtRW5c65M997T0Py09RFsvYdHFjho1MBI--6qKIzm42ugZnTPTI79Er2hPVMfA2JDxY6u-rI-adO1wR4pyxT3G4A9HGh4UD3q9VwWtfJJAL7b6vG-7Iru2G9k13xFITU2At7LJbE56IiqV9PFKHob3LIGSqJaT87gEEy0WI6BTE8rfLz-fdCfJYjrSL_fql7wW4EbxkqBKblvRyusm1Uqtf30r44VUD2l3kBwMEGmFdUgCDa5l5Pd3UeEx-mGQ2COK74j-PTMT3XTShbgIlgzGanu7uLGfe-ZambXkdHS9VrbhRb5mihcveTzo26mpaHx5pCzcfvOnnz2Xw3ietggKXyYbnyA2I8ue4K-e7JGMRtxnSBWRPVbPGmR8uQWx0cRnwcWvmKPRpBpU-1V-rmK0BJ79qPcP55nUQvL3XZRi-ET4ZycL6bRKXq0qxNzSdMcB0CCrjw3tZDxgPeQ3IHsHGoFYa9aKOghCsfpL_GLSwS1p84OK6Xjy-lqCGkSszGVr3x_j0kngNB7iyOg2sg8MFT8Ab8PUF8EJBC0qYIw7XqRIqao-e9IJaP-jonUlgZUdtzA4k1yWvWTLS05eUP_3OUtAS4sSC0")

# Local audio directory (where your samples are)
LOCAL_AUDIO_DIR = Path("./audio_data/")  # change to your path

# Audio extensions to show in UI
AUDIO_EXTS = (".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac")

## 2) Imports & Utilities
We keep helpers small and focused.

In [3]:
# === IMPORTS & UTILITIES ===
import requests
from typing import List
from IPython.display import Audio

def list_local_audio_files(directory: Path, exts=AUDIO_EXTS) -> List[Path]:
    directory = Path(directory)
    return sorted([p for p in directory.glob("*") if p.suffix.lower() in exts and p.is_file()])

## 3) Transcription function (HTTP via `requests`)
Sends a **multipart/form-data** POST to the Whisper endpoint. Returns the transcript text when available.

In [4]:
def transcribe_with_whisper(
    local_audio_path: Path,
    model_name: str = WHISPER_MODEL,
    api_url: str = WHISPER_API,
    api_key: str | None = OPENAI_API_KEY,
    timeout: int = 180,
):
    """Send a local file to an OpenAI-compatible Whisper endpoint and return transcript text (or raw JSON)."""

    headers = {}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"

    # Prepare file and form data
    with open(local_audio_path, "rb") as f:
        files = {"file": (local_audio_path.name, f, "application/octet-stream")}
        data = {"model": model_name}

        # --- Folder-based language override ---
        parent_folder = local_audio_path.parent.name.lower()
        if "danish" in parent_folder:
            data["language"] = "da"
            print(f"🌍 Detected Danish folder — forcing language='da'")
        elif "english" in parent_folder and "language" in data:
            data.pop("language", None)
            print(f"🌍 Detected English folder — removing language param")

        # --- Perform HTTP request safely ---
        try:
            resp = requests.post(api_url, headers=headers, files=files, data=data, timeout=timeout)
            resp.raise_for_status()
        except Exception as e:
            print(f"⚠️ Transcription request failed: {e}")
            if 'resp' in locals():
                print("Server response:", getattr(resp, "text", "(no text)"))
            raise

    # --- Parse response ---
    try:
        js = resp.json()
    except Exception:
        print("⚠️ Failed to parse JSON response. Returning raw text instead.")
        return resp.text

    if "text" in js:
        return js["text"]
    if isinstance(js, dict) and "choices" in js and js["choices"]:
        choice = js["choices"][0]
        if isinstance(choice, dict) and "text" in choice:
            return choice["text"]

    return js  # fallback if the server returns a different structure

## 4) UI — Pick, play, and transcribe
Use **Refresh** to scan the folder, **Play** to preview, and **Transcribe** to send the file to Whisper.

In [5]:
# === UI WIDGETS ===
from IPython.display import display, clear_output
import ipywidgets as widgets
from pathlib import Path

# --- Imports ---
import ipywidgets as widgets
from pathlib import Path
from IPython.display import display, clear_output, Audio

# --- Inputs ---
# Base audio folder (root path where subfolders exist)
BASE_AUDIO_DIR = Path("audio_data")

# Dynamically detect available subfolders (e.g., english, danish, etc.)
subfolders = [
    f.name for f in BASE_AUDIO_DIR.iterdir()
    if f.is_dir()
    and not f.name.startswith('.')               # ignore hidden dirs
    and f.name != '.ipynb_checkpoints'           # explicitly ignore checkpoints
]
folder_dd = widgets.Dropdown(
    options=subfolders,
    value=subfolders[0] if subfolders else None,
    description="Folder:",
    layout=widgets.Layout(width="40%")
)

refresh_btn = widgets.Button(description="Refresh", icon="refresh")
file_dd = widgets.Dropdown(options=[], description="File:", layout=widgets.Layout(width="70%"))

# Actions
play_btn = widgets.Button(description="Play", icon="play")
transcribe_btn = widgets.Button(description="Transcribe", icon="microphone")

# Outputs
status_out = widgets.Output()
audio_out = widgets.Output()
text_out = widgets.Output()

# --- Helper functions ---
def list_local_audio_files(folder: Path):
    return [str(p) for p in folder.glob("*.wav")]

def refresh_files(_=None):
    """Refresh file list based on selected folder."""
    if not folder_dd.value:
        return
    folder = BASE_AUDIO_DIR / folder_dd.value
    files = list_local_audio_files(folder)
    file_dd.options = files
    with status_out:
        clear_output(wait=True)
        if files:
            print(f"Found {len(files)} audio file(s) in {folder}")
        else:
            print(f"No audio files found in {folder}")

def play_audio(_=None):
    sel = file_dd.value
    if not sel:
        return
    with audio_out:
        clear_output(wait=True)
        display(Audio(filename=str(sel), autoplay=False))

def run_transcription(_=None):
    sel = file_dd.value
    if not sel:
        return
    with status_out:
        clear_output(wait=True)
        print(f"Transcribing: {Path(sel).name}")
    try:
        txt = transcribe_with_whisper(Path(sel))
        with text_out:
            clear_output(wait=True)
            print("=== Transcript (HTTP Request)===")
            print(txt if isinstance(txt, str) else str(txt))
        with status_out:
            clear_output(wait=True)
            print("Done.")
    except Exception as e:
        with text_out:
            clear_output(wait=True)
            print("Transcription failed (HTTP Request):", e)

refresh_btn.on_click(refresh_files)
play_btn.on_click(play_audio)
transcribe_btn.on_click(run_transcription)

# --- Folder change updates file list automatically ---
def on_folder_change(change):
    refresh_files()

folder_dd.observe(on_folder_change, names="value")

# Render the UI
display(widgets.HBox([folder_dd, refresh_btn]))
display(file_dd)
display(widgets.HBox([play_btn, transcribe_btn]))
display(status_out, audio_out, text_out)

# initial file load
refresh_files()

Dropdown(description='File:', layout=Layout(width='70%'), options=(), value=None)

Output()

Output()

Output()

## 5) (Optional) Quick smoke test
Runs transcription on the currently selected file (or the first file in the folder if nothing is selected).

In [6]:
# --- Smoke test ---
from pathlib import Path

if file_dd.value:
    test_file = Path(file_dd.value)
else:
    folder = BASE_AUDIO_DIR / (folder_dd.value or "")
    files = list_local_audio_files(folder)
    test_file = Path(files[0]) if files else None

if test_file and test_file.exists():
    print(f"🎤 Transcribing via HTTP request: {test_file.name}")
    try:
        txt = transcribe_with_whisper(test_file)
        print("=== Transcript ===")
        print(txt if isinstance(txt, str) else str(txt))
    except Exception as e:
        print("⚠️ Transcription failed:", e)
else:
    print("❌ No audio file selected or found for quick test.")


🎤 Transcribing via HTTP request: audio_01.wav
🌍 Detected Danish folder — forcing language='da'
=== Transcript ===
 Så mister man sin stemmeret i otte år.


## 6) Troubleshooting
- **Port & reachability**: Confirm `:8080` and that this notebook can reach `whisper-...svc.cluster.local`.
- **Auth**: If your gateway requires a token, export `OPENAI_API_KEY` and re-run the config cell.
- **Server errors**: Consider printing `resp.text` on non-2xx responses inside `transcribe_with_whisper` for more detail.
- **Large files**: Increase `timeout` in `transcribe_with_whisper` as needed.